# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','onnxsim':'onnxsim','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import collections, shutil
import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
from onnxsim import simplify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 63.8 MB/s eta 0:00:00


In [4]:

TASK_ID = 'task158'
CH = 10
H = W = 30
COMPETITION = Path('/kaggle/input/competitions/neurogolf-2026')
ROOT =  Path.cwd()
MODEL_PATH = ROOT / f'{TASK_ID}.onnx'
ZIP_PATH = ROOT / 'submission.zip'
print('ROOT:', ROOT)
print('MODEL_PATH:', MODEL_PATH)

ROOT: /kaggle/working
MODEL_PATH: /kaggle/working/task158.onnx


In [5]:
# Load task data. Kaggle path first, local /mnt/data fallback for offline verification.
task_path = COMPETITION / f'{TASK_ID}.json'
if not task_path.exists():
    task_path = Path('/mnt/data') / f'{TASK_ID}.json'
assert task_path.exists(), f'Missing task json: {task_path}'
task = json.load(open(task_path))
print('task_path:', task_path)
print({k: len(v) for k, v in task.items()})

task_path: /kaggle/input/competitions/neurogolf-2026/task158.json
{'train': 3, 'test': 1, 'arc-gen': 262}


In [6]:

# -----------------------------
# Model construction
# -----------------------------
# The rule implemented here:
# 1. Detect the background as the most frequent color.
# 2. Detect the 3x3 reference pattern: a local patch containing >=3 non-background colors.
# 3. Infer the fill color from the densest non-background color in that reference patch.
# 4. Detect isolated square markers of size 1x1, 2x2, or 3x3.
# 5. Match opposite-corner marker-color pairs to the reference's opposite-corner marker-color pair,
#    allowing the eight dihedral transforms of a 3x3 motif.
# 6. Draw the transformed fill-color motif into the matched target square.
#
# Implementation constraint:
# Drawing is done by compile-time fixed Slice + Concat + Add, not by F.pad or ConvTranspose.

corners = [(0,0), (0,2), (2,0), (2,2)]
opposite = {(0,0):(2,2), (2,2):(0,0), (0,2):(2,0), (2,0):(0,2)}

def tf(coord, t):
    r, c = coord
    if t == 0: return (r, c)          # identity
    if t == 1: return (c, 2-r)        # rot90
    if t == 2: return (2-r, 2-c)      # rot180
    if t == 3: return (2-c, r)        # rot270
    if t == 4: return (r, 2-c)        # horizontal mirror
    if t == 5: return (2-r, c)        # vertical mirror
    if t == 6: return (c, r)          # main diagonal mirror
    if t == 7: return (2-c, 2-r)      # anti-diagonal mirror
    raise ValueError(t)

def find_t(pa, pb, ta, tb):
    for t in range(8):
        if tf(pa, t) == ta and tf(pb, t) == tb:
            return t
    return None

class Task158ConstructedStatic(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('w3', torch.ones(CH, 1, 3, 3))
        for s in [1, 2, 3]:
            self.register_buffer(f'w{s}', torch.ones(CH, 1, s, s))
            ring = torch.ones(CH, 1, s+2, s+2)
            ring[:, :, 1:s+1, 1:s+1] = 0
            self.register_buffer(f'ring{s}', ring)

        # Static zero buffers for the one-cell border used by isolated-square detection.
        # These replace F.pad(nonbg, (1,1,1,1)).
        self.register_buffer('z_col30', torch.zeros(1, 1, 30, 1))
        self.register_buffer('z_row32', torch.zeros(1, 1, 1, 32))

        # Static zero buffers for drawing target squares without ConvTranspose or F.pad.
        # For s=1,2,3, the detector map has side M = 30 - 3*s + 1 = 28,25,22.
        # The maximum fixed padding amount is 3*s-1 <= 8.
        for M in [28, 25, 22]:
            for k in range(1, 9):
                self.register_buffer(f'zdL_{M}_{k}', torch.zeros(1, 1, M, k))
                self.register_buffer(f'zdT_{M}_{k}', torch.zeros(1, 1, k, 30))

    def color_onehot_from_idx(self, idx):
        return torch.cat([(idx == k).float() for k in range(CH)], dim=1)  # B,C,1,1

    def pad1_static(self, t):
        # t: B,1,30,30 -> B,1,32,32 without F.pad / ConstantOfShape.
        t = torch.cat([self.z_col30, t, self.z_col30], dim=3)
        t = torch.cat([self.z_row32, t, self.z_row32], dim=2)
        return t

    def square_det_all(self, x, nonbg, s: int):
        cnt = F.conv2d(x, getattr(self, f'w{s}'), groups=CH)
        ring_count = F.conv2d(self.pad1_static(nonbg), getattr(self, f'ring{s}')[:1])
        return (cnt == float(s*s)).float() * (ring_count == 0).float()

    def place_det(self, det, top: int, left: int, M: int):
        # det: B,1,M,M. Place it into a 30x30 canvas by fixed Concat only.
        right = 30 - M - left
        bottom = 30 - M - top

        parts = []
        if left > 0:
            parts.append(getattr(self, f'zdL_{M}_{left}'))
        parts.append(det)
        if right > 0:
            parts.append(getattr(self, f'zdL_{M}_{right}'))
        y = torch.cat(parts, dim=3) if len(parts) > 1 else parts[0]

        parts = []
        if top > 0:
            parts.append(getattr(self, f'zdT_{M}_{top}'))
        parts.append(y)
        if bottom > 0:
            parts.append(getattr(self, f'zdT_{M}_{bottom}'))
        y = torch.cat(parts, dim=2) if len(parts) > 1 else parts[0]
        return y

    def draw_square(self, det, s: int, dr: int, dc: int):
        # Equivalent to ConvTranspose with an s-by-s ones block, then fixed pad to 30x30.
        # Implemented as sum of shifted detector maps, so the exported ONNX uses only basic ops.
        M = 30 - 3*s + 1
        pieces = []
        for rr in range(s):
            for cc in range(s):
                pieces.append(self.place_det(det, dr + rr, dc + cc, M))
        out = pieces[0]
        for p in pieces[1:]:
            out = out + p
        return out.clamp(0, 1)

    def forward(self, x):
        active = (x.sum(1, keepdim=True) > 0).float()
        counts = x.sum((2, 3), keepdim=True)
        bg_idx = counts.argmax(1, keepdim=True)
        bg_one = self.color_onehot_from_idx(bg_idx)

        c3 = F.conv2d(x, self.w3, groups=CH)  # B,C,28,28
        distinct = ((c3 > 0).float() * (1 - bg_one)).sum(1, keepdim=True)
        mult = (distinct >= 3).float()

        scores = (c3 * mult * (1 - bg_one)).amax((2, 3), keepdim=True)
        fill_idx = scores.argmax(1, keepdim=True)
        fill_one = self.color_onehot_from_idx(fill_idx)
        fill_mask = (x * fill_one).sum(1, keepdim=True)

        fill_count_in_patch = (c3 * fill_one).sum(1, keepdim=True)
        fill_score = (scores * fill_one).sum(1, keepdim=True)
        ref_det = (fill_count_in_patch == fill_score).float() * mult * (fill_score > 0).float()

        bgmask = (x * bg_one).sum(1, keepdim=True)
        nonbg = (active - bgmask).clamp(0, 1)
        sq1 = self.square_det_all(x, nonbg, 1)
        sq2 = self.square_det_all(x, nonbg, 2)
        sq3 = self.square_det_all(x, nonbg, 3)

        fill_add = x * 0.0
        nonmarker_color_mask = (1 - bg_one) * (1 - fill_one)

        for pa in corners:
            pb = opposite[pa]
            va = (ref_det * x[:, :, pa[0]:pa[0]+28, pa[1]:pa[1]+28]).sum((2, 3), keepdim=True) * nonmarker_color_mask
            vb = (ref_det * x[:, :, pb[0]:pb[0]+28, pb[1]:pb[1]+28]).sum((2, 3), keepdim=True) * nonmarker_color_mask
            valid = (va.sum(1, keepdim=True) > 0).float() * (vb.sum(1, keepdim=True) > 0).float()
            valid = valid * (1 - (va * vb).sum(1, keepdim=True).clamp(0, 1))

            refbits = []
            for rr in range(3):
                for cc in range(3):
                    bit = ((ref_det * fill_mask[:, :, rr:rr+28, cc:cc+28]).sum((2, 3), keepdim=True) > 0).float() * valid
                    refbits.append(bit)

            for ta in corners:
                tb = opposite[ta]
                tr = find_t(pa, pb, ta, tb)
                if tr is None:
                    continue

                for s, sd in [(1, sq1), (2, sq2), (3, sq3)]:
                    M = H - 3*s + 1
                    Apos = (ta[0] * s, ta[1] * s)
                    Bpos = (tb[0] * s, tb[1] * s)
                    Ad = (sd * va).sum(1, keepdim=True)[:, :, Apos[0]:Apos[0]+M, Apos[1]:Apos[1]+M]
                    Bd = (sd * vb).sum(1, keepdim=True)[:, :, Bpos[0]:Bpos[0]+M, Bpos[1]:Bpos[1]+M]
                    pair = Ad * Bd * valid

                    idx = 0
                    for rr in range(3):
                        for cc in range(3):
                            bit = refbits[idx]
                            idx += 1
                            trr, tcc = tf((rr, cc), tr)
                            det = pair * bit
                            pix = self.draw_square(det, s, trr*s, tcc*s)
                            fill_add = (fill_add + pix * fill_one).clamp(0, 1)

        total_fill = fill_add.sum(1, keepdim=True).clamp(0, 1) * active
        return (x * (1 - total_fill) + fill_add).clamp(0, 1)


In [7]:
# Export ONNX from the constructed model.
torch.manual_seed(0)
model = Task158ConstructedStatic().eval()
with torch.no_grad():
    torch.onnx.export(
        model,
        torch.zeros(1, CH, H, W),
        str(MODEL_PATH),
        input_names=['input'],
        output_names=['output'],
        opset_version=13,
        dynamo=False,
    )
print('onnx:', MODEL_PATH, MODEL_PATH.stat().st_size, 'bytes')


/tmp/ipykernel_17/3699233403.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


onnx: /kaggle/working/task158.onnx 813866 bytes


In [8]:
# ONNX graph safety validation.
m = onnx.load(str(MODEL_PATH))
onnx.checker.check_model(m)
ops = collections.Counter(node.op_type for node in m.graph.node)
print('ops:', dict(sorted(ops.items())))

shape_in = [d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim]
shape_out = [d.dim_value for d in m.graph.output[0].type.tensor_type.shape.dim]
print('input shape:', shape_in)
print('output shape:', shape_out)

forbidden = ['Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function']
hard_risk = ['ScatterND', 'Shape', 'Range', 'Expand', 'Gather', 'ConstantOfShape', 'Pad', 'ConvTranspose']
bad = [op for op in forbidden + hard_risk if ops.get(op, 0)]
print('bad ops:', bad)
assert shape_in == [1, 10, 30, 30]
assert shape_out == [1, 10, 30, 30]
assert MODEL_PATH.stat().st_size < 1_400_000
assert not bad


ops: {'Add': 2017, 'ArgMax': 2, 'Cast': 44, 'Clip': 871, 'Concat': 2884, 'Constant': 2155, 'Conv': 6, 'Equal': 27, 'Greater': 16, 'GreaterOrEqual': 1, 'Identity': 7, 'Mul': 1053, 'ReduceMax': 1, 'ReduceSum': 41, 'Slice': 90, 'Sub': 8}
input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
bad ops: []


In [9]:
# Exact-match validation on visible examples and arc-gen.
opts = ort.SessionOptions()
opts.intra_op_num_threads = 1
opts.inter_op_num_threads = 1
sess = ort.InferenceSession(str(MODEL_PATH), sess_options=opts, providers=['CPUExecutionProvider'])

def grid_to_tensor(grid):
    a = np.array(grid, dtype=np.int64)
    h, w = a.shape
    x = np.zeros((1, CH, H, W), dtype=np.float32)
    for k in range(CH):
        x[0, k, :h, :w] = (a == k)
    return x

def predict(grid):
    h, w = len(grid), len(grid[0])
    y = sess.run(None, {'input': grid_to_tensor(grid)})[0]
    return y[0, :, :h, :w].argmax(0).astype(np.int64).tolist()

def exact(ex):
    return predict(ex['input']) == ex['output']

for split in ['train', 'test', 'arc-gen']:

    ok = sum(exact(ex) for ex in task[split])
    total = len(task[split])

    assert ok == total

idx = list(range(len(task['arc-gen'])))
random.Random(0).shuffle(idx)
test_idx = set(idx[:round(0.3 * len(idx))])
fit_ok = fit_total = test_ok = test_total = 0
for i, ex in enumerate(task['arc-gen']):
    ok = exact(ex)
    if i in test_idx:
        test_ok += ok; test_total += 1
    else:
        fit_ok += ok; fit_total += 1
print(f'70/30 arc-gen split: fit {fit_ok}/{fit_total}, test {test_ok}/{test_total}')
assert fit_ok == fit_total and test_ok == test_total


70/30 arc-gen split: fit 183/183, test 79/79


In [10]:
# Package Kaggle submission.
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH, f'{TASK_ID}.onnx')
    
print('submission:', ZIP_PATH, ZIP_PATH.stat().st_size, 'bytes')

with zipfile.ZipFile(ZIP_PATH) as z:
    print('zip contents:', z.namelist())


submission: /kaggle/working/submission.zip 105324 bytes
zip contents: ['task158.onnx']
